In [2]:
import pandas as pd
import math

# Параметры из фичи (задаются в начале)
Z_SCORE = 1.96
PRAG_EROARE = 0.10
MARJA_DORITA = 0.05

# 1. DEFINIRE
print("PASUL 1 — Definire (Cartea de identitate)")
cartea_identitate = {
    "Scop": "Evaluarea numărului de angajați în industria alimentară pe tipuri de producție",
    "Sursă": "Registrul intern al producătorilor alimentari, versiunea 2024",
    "Țintă": "NACE Rev. 2 (CAEM-2 în Republica Moldova), versiunea 2.1",
    "Nivel": "Clasă (4 cifre)",
    "Perioadă": "Anul 2024"
}
for cheie, valoare in cartea_identitate.items():
    print(f"  {cheie}: {valoare}")


# 2. INVENTAR
print("\nPASUL 2 — Inventar")
cadru_date = pd.read_csv("Lab6.csv", dtype={'cod_local': str})

total_firme = cadru_date['firme'].sum()
total_angajati = cadru_date['angajati'].sum()

angajati_clari = total_angajati
pondere_clara = 100 * angajati_clari / total_angajati

print(cadru_date.to_string(index=False))
print(f"\n  Total firme: {total_firme}")
print(f"  Total angajați: {total_angajati}")
print(f"  Ponderea angajaților din coduri clare: {pondere_clara:.1f}%")
print(f"  Coduri neclare: nu există (toate cele 7 coduri L1-L7 au descriere)")


# 3. CANDIDAȚI
print("\nPASUL 3 — Candidați (validarea codurilor țintă)")
mapare_nace = {
    'L1': '10.71', 
    'L2': '11.02', 
    'L3': '10.61', 
    'L4': '10.62', 
    'L5': '10.51', 
    'L6': '10.39', 
    'L7': '10.89'  
}

for cod_local, cod_nace in mapare_nace.items():
    print(f"  {cod_local} -> {cod_nace} (valid)")


# 4. TIPUL LEGĂTURII
print("\nPASUL 4 — Tipul legăturii (1:1, 1:n, nemapat)")
legaturi = {
    'L1': ('10.71', '1:1'),
    'L2': ('11.02', '1:1'),
    'L3': ('10.61', '1:1'),
    'L4': ('10.62', '1:1'),
    'L5': ('10.51', '1:n'),
    'L6': ('10.39', '1:n'),
    'L7': ('10.89', '1:1')
}

for cod_local, (cod_nace, tip) in legaturi.items():
    print(f"  {cod_local} -> {cod_nace} ({tip})")

total_coduri = len(legaturi)
clare_count = sum(1 for _, tip in legaturi.values() if tip != 'nemapat')
print(f"  Procent coduri clare: {100 * clare_count / total_coduri:.1f}%")

print("  Sens invers:")
clase_nace = set(v[0] for v in legaturi.values())
for cod_nace in clase_nace:
    surse = [k for k, v in legaturi.items() if v[0] == cod_nace]
    print(f"    {cod_nace} <- {', '.join(surse)}")


# 5. PONDERI (p̂, SE, IC 95%)
print("\nPASUL 5 — Ponderi și IC 95% pentru codurile 1:n")
esantioane = {
    'L5': {'n': 40, 'x': 24, 'clasa_tinta': '10.51'},
    'L6': {'n': 50, 'x': 35, 'clasa_tinta': '10.39'}
}

for cod_local, date_esantion in esantioane.items():
    p = date_esantion['x'] / date_esantion['n']
    eroare_standard = math.sqrt(p * (1 - p) / date_esantion['n'])
    ic_min = p - Z_SCORE * eroare_standard
    ic_max = p + Z_SCORE * eroare_standard
    
    print(f"  {cod_local} -> {date_esantion['clasa_tinta']}:")
    print(f"    p̂ = {p:.4f} ({p*100:.1f}%)")
    print(f"    SE = {eroare_standard:.4f}")
    print(f"    IC 95% = [{ic_min:.4f}, {ic_max:.4f}]")
    
    complement = 1 - p
    print(f"    Pondere complementară = {complement:.4f}")


# 6. TABEL FINAL ȘI MAPARE
print("\nPASUL 6 — Tabel final și mapare")
cadru_date['nace'] = cadru_date['cod_local'].map(mapare_nace)

alocare = cadru_date.groupby('nace')['angajati'].sum().reset_index()

total_mapat = cadru_date[cadru_date['nace'].notna()]['angajati'].sum()
total_nemapat = cadru_date[cadru_date['nace'].isna()]['angajati'].sum()

print(alocare.to_string(index=False))
print(f"\n  Control: mapat ({total_mapat}) + nemapat ({total_nemapat}) = {total_mapat + total_nemapat}")
print(f"  Total inițial: {total_angajati}")

fisier_urma = cadru_date[['cod_local', 'descriere', 'nace', 'angajati']].copy()
fisier_urma['versiune'] = '1.0'
fisier_urma.to_csv("urma_mapare.csv", index=False)
print("  Fișierul 'urma_mapare.csv' a fost salvat.")


# 7. VALIDARE ȘI ACTUALIZARE
print("\nPASUL 7 — Validare și actualizare")
n_validare = 100
x_validare = 7
rata_eroare = x_validare / n_validare

print(f"  Rata de eroare: {rata_eroare*100:.1f}% (prag: {PRAG_EROARE*100:.0f}%)")
if rata_eroare <= PRAG_EROARE:
    print("  Decizie: ACCEPTAT")
else:
    print("  Decizie: RESPINS")

p_estimat = rata_eroare
n_necesar = (Z_SCORE**2 * p_estimat * (1 - p_estimat)) / (MARJA_DORITA**2)
print(f"  Eșantion necesar pentru marja ±{MARJA_DORITA*100:.0f}%: {math.ceil(n_necesar)} unități")

print("  Cod nou L8 (berea artizanală) rămâne NEMAPAT.")
print("  Versiunea tabelului crește: 1.0 -> 1.1")

PASUL 1 — Definire (Cartea de identitate)
  Scop: Evaluarea numărului de angajați în industria alimentară pe tipuri de producție
  Sursă: Registrul intern al producătorilor alimentari, versiunea 2024
  Țintă: NACE Rev. 2 (CAEM-2 în Republica Moldova), versiunea 2.1
  Nivel: Clasă (4 cifre)
  Perioadă: Anul 2024

PASUL 2 — Inventar
 id cod_local                               descriere  firme  angajati
  1        L1                panificatie si patiserie      9        70
  2        L2                             vinificatie      4        45
  3        L3                    macinarea cerealelor      3        20
  4        L4             productia de faina si crupe      4        30
  5        L5            produse lactate si inghetata      5        60
  6        L6 conserve si sucuri din fructe si legume      6        50
  7        L7                 alte produse alimentare      3        15

  Total firme: 34
  Total angajați: 290
  Ponderea angajaților din coduri clare: 100.0%
  Coduri n